# Practical 11: Search a Static Website

**Aim:** Read local HTML pages, build a word-to-page index, and rank pages for a search query.

This uses Python's built-in HTML parser and needs no extra packages. The example pages are in `website/`. Run the notebook with this practical folder as the working directory, then run cells from top to bottom.


## 1. Read and index the website pages

The parser collects words from each HTML page. The inverted index maps every word to its pages and count.


In [ ]:
from collections import Counter
from html.parser import HTMLParser
from pathlib import Path
import re

SITE = Path("website")

class TextParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.words = []

    def handle_data(self, text):
        self.words.extend(re.findall(r"[a-z]+", text.lower()))

pages = {}
for file in SITE.rglob("*.html"):
    parser = TextParser()
    parser.feed(file.read_text(encoding="utf-8"))
    pages[file] = Counter(parser.words)

index = {}
for file, words in pages.items():
    for word, count in words.items():
        index.setdefault(word, {})[file] = count


## 2. Search

Change `query` to try another search. Pages with more query word matches receive higher scores.


In [ ]:
query = "machine learning"
scores = {}
for word in re.findall(r"[a-z]+", query.lower()):
    for file, count in index.get(word, {}).items():
        scores[file] = scores.get(file, 0) + count

for file, score in sorted(scores.items(), key=lambda item: item[1], reverse=True):
    print(score, file.relative_to(SITE))
if not scores:
    print("No matching pages")


## Viva questions with answers

1. **What is a static website?** A set of HTML pages served as written rather than generated for each request.
2. **What does this search engine search?** Text inside local HTML files in the `website/` folder.
3. **What is an inverted index?** A mapping from each word to the pages that contain it.
4. **What is term frequency?** The number of times a word appears in one page.
5. **How are results ranked?** Query word counts are added for each page, then pages are sorted by score.
6. **What does `HTMLParser` do?** It reads HTML and passes text between tags to the program.
7. **Why lowercase words?** So differently capitalized versions match the same index term.
8. **What is the crawler here?** The part that visits each local HTML file and extracts its text.
9. **What if a query word is missing from the index?** It adds no score to any page.
10. **How can the index cover more pages?** Add HTML files under `website/`; the program finds them recursively.
